# Phase 2

1. Loads the PhiUSIIL phishing URL dataset and has a quick look at it (Task 1).
2. Does the bare minimum of cleaning so the eLCS code can read the data.
3. Trains the *original, unmodified* eLCS with its default settings and checks how well it does (Task 2).

**S1** uses all 50 numeric features, **S2** removes 4 "derived" features that might leak the answer.
Phishing is treated as the "positive" class, so *recall* = how many phishing sites we catch.

In [1]:
import sys, time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

from skeLCS import eLCS                              # the original eLCS, nothing inside it is changed

CSV = "PhiUSIIL_Phishing_URL_Dataset.csv"            # the dataset file
SEED = 0                                             # makes the split and the eLCS run repeatable

## 1. Looking at the data (Task 1)

In [2]:
df = pd.read_csv(CSV)
print("rows, columns:", df.shape)
print()
print("label counts (1 = legitimate, 0 = phishing):")
print(df["label"].value_counts())
print()
print("missing values:", int(df.isna().sum().sum()))
print("rows that repeat an earlier URL:", int(df["URL"].duplicated().sum()))
df.head(3)

rows, columns: (235795, 56)

label counts (1 = legitimate, 0 = phishing):
label
1    134850
0    100945
Name: count, dtype: int64

missing values: 0
rows that repeat an earlier URL: 425


,FILENAME,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,...,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,521848.txt,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,...,0,0,1,34,20,28,119,0,124,1
1,31372.txt,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,...,0,0,1,50,9,8,39,0,217,1
2,597387.txt,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,...,0,0,1,10,2,7,42,2,5,1


**What we see (Phase I recap)**
- About 236k pages, 56 columns: 50 numeric features, 5 text/ID columns (`FILENAME`, `URL`, `Domain`, `TLD`, `Title`) and the `label`.
- Nothing is missing. A few hundred rows repeat a URL we've already seen.
- Classes are a bit lopsided (about 57% legitimate, 43% phishing), but not badly.

## 2. Minimal cleaning
eLCS only understands numbers, so the 5 text columns have to go. Everything else stays untouched
(no scaling, no outlier removal) because Task 2 is about the original eLCS on raw data.

The one thing we do is remove repeated URLs, so the same page can't end up in both the training and test sets.

In [3]:
TEXT_COLS = ["FILENAME", "URL", "Domain", "TLD", "Title"]
# 4 features that are calculated from dataset-wide numbers so they could leak the answer.
# (e.g. URLSimilarityIndex is exactly 100 for every legitimate URL.)
DERIVED = ["URLSimilarityIndex", "TLDLegitimateProb", "URLCharProb", "CharContinuationRate"]

df = df.drop_duplicates(subset="URL")                  # keep the first copy of each URL
y = (df["label"] == 0).astype(int).values              # 1 = phishing (the "positive" class), 0 = legitimate
X_all = df.drop(columns=TEXT_COLS + ["label"])         # numeric features only

print("rows after removing repeated URLs:", len(df))
print("numeric features:", X_all.shape[1])

rows after removing repeated URLs: 235370
numeric features: 50


## 3. Split into train and test
We hold back 20% of the rows for testing (stratified, so both sets keep the same phishing/legitimate mix).
eLCS never sees the test rows while learning.

In [4]:
X_train_all, X_test_all, y_train, y_test = train_test_split(
    X_all, y, test_size=0.2, stratify=y, random_state=SEED)
print("train rows:", len(X_train_all), "| test rows:", len(X_test_all))

train rows: 188296 | test rows: 47074


## 4. Training the original eLCS (Task 2)
All settings are the package defaults, only set learning_iterations=10000 (the default) and a random seed.

Two things to know about how eLCS works:
- It reads the training rows in the order we give them and uses one row per learning iteration,
  so with 10,000 iterations it only ever learns from 10,000 rows. We shuffle first so those 10,000 are random.
- Its model is a long list of IF-THEN rules. A prediction is a vote between all rules that match the new row.

In [ ]:
def run_original_elcs(setting):
    #Train + test the default eLCS. setting is 'S1' (all features) or 'S2' (derived features removed)."""
    cols = list(X_all.columns) if setting == "S1" else [c for c in X_all.columns if c not in DERIVED]
    Xtr, Xte = X_train_all[cols].values, X_test_all[cols].values

    order = np.random.RandomState(SEED).permutation(len(Xtr))   # shuffle the training rows
    model = eLCS(learning_iterations=10000, random_state=SEED)  # everything else = package defaults

    t0 = time.time()
    model.fit(Xtr[order], y_train[order])                       # learn the rules
    fit_seconds = time.time() - t0

    pred = model.predict(Xte)                                   # slow-ish: every rule is checked for every row

    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    rules = model.population.popSet                             # the learned rules
    return {
        "setting": setting,
        "accuracy %": 100 * accuracy_score(y_test, pred),
        "precision %": 100 * precision_score(y_test, pred),
        "recall %": 100 * recall_score(y_test, pred),
        "F1 %": 100 * f1_score(y_test, pred),
        "errors": int(fp + fn),
        "missed phishing": int(fn),       # phishing sites we called legitimate (the costly mistake)
        "false alarms": int(fp),          # legitimate sites we called phishing
        "rules": len(rules),
        "conditions per rule": float(np.mean([len(r.specifiedAttList) for r in rules])),
        "train seconds": fit_seconds,
    }

In [ ]:
results = [run_original_elcs("S1"), run_original_elcs("S2")]   # predicting takes a few minutes
table = pd.DataFrame(results).set_index("setting").round(3)
table

,accuracy %,precision %,recall %,F1 %,errors,missed phishing,false alarms,rules,conditions per rule,train seconds
setting,,,,,,,,,,
S1,99.762,99.544,99.901,99.722,112,20,92,945,16.750,40.987
S2,98.827,98.386,98.876,98.631,552,226,326,948,16.037,34.650


## 5. What the results say
- **S1 (all features):** the default eLCS is already very accurate, because URLSimilarityIndex alone almost gives the answer away.
- **S2 (derived features removed):** accuracy drops and there are many more mistakes, so this is the more realistic test.
- The final model holds roughly **940 rules with about 16 conditions each**.
- This is **one split** with one random seed, so the numbers wobble a little from run to run.
  In the report we repeated this 10 times and averaged (S1: 99.86% accuracy, S2: 98.79% accuracy).

## 6. Preprocessing and feature engineering (Task 3)
Here I'm doing three things to make its job easier:
1. **Data checks:** missing values, impossible values and duplicates.
2. **Scaling:** squash every feature onto a 0-1 scale, so huge outliers don't break eLCS's ranges.
3. **Feature selection:** keep only the most useful features, so the rules are shorter.

Everything is learned from the training rows only and then applied to the test rows, so the test set stays unseen.

### Data checks

In [ ]:
print("missing values:", int(X_all.isna().sum().sum()))      # 0 = nothing to fill in
print("negative values:", int((X_all < 0).sum().sum()))      # 0 = nothing impossible
print("duplicate URLs left:", int(df["URL"].duplicated().sum()))   # 0 = removed back in section 2
print("LineOfCode  median:", X_all["LineOfCode"].median(), "| max:", X_all["LineOfCode"].max())   # outliers

missing values: 0
negative values: 0
duplicate URLs left: 0
LineOfCode  median: 431.0 | max: 442666


There are no missing or impossible values and the duplicate URLs are already gone. LineOfCode shows the outlier problem: the typical page has a few hundred lines, the biggest has hundreds of thousands. So keep those rows (they are real pages) and let the scaling deal with them.

### Scaling and feature selection
- **Scaling:** QuantileTransformer replaces every value with its rank between 0 and 1. The 0/1 flags stay exactly 0 and 1.
- **Feature selection:** *mutual information* measures how much a feature tells us about phishing vs legitimate.
  We keep the top k (15 for S1 and 30 for S2).

In [ ]:
from sklearn.preprocessing import QuantileTransformer
from sklearn.feature_selection import mutual_info_classif

def preprocess(setting, k):
    # S1 = all features, S2 = without the 4 derived ones
    cols = list(X_all.columns) if setting == "S1" else [c for c in X_all.columns if c not in DERIVED]

    # scaling: learn the ranks from the TRAIN rows, then apply them to both train and test
    scaler = QuantileTransformer(n_quantiles=1000, random_state=SEED).fit(X_train_all[cols])
    Xtr = scaler.transform(X_train_all[cols])
    Xte = scaler.transform(X_test_all[cols])

    # feature selection: score each feature on a 20,000-row sample of the training data (keeps it quick)
    sample = np.random.RandomState(SEED).choice(len(Xtr), 20000, replace=False)
    scores = mutual_info_classif(Xtr[sample], y_train[sample], random_state=SEED)
    best = np.argsort(scores)[::-1][:k]                      # positions of the k highest scores
    kept = [cols[i] for i in best]
    return Xtr[:, best], Xte[:, best], kept

prepared = {}
for setting, k in [("S1", 15), ("S2", 30)]:
    Xtr_p, Xte_p, kept = preprocess(setting, k)
    prepared[setting] = (Xtr_p, Xte_p)
    print(setting, "-> kept", len(kept), "features:", kept)
    print("   train", Xtr_p.shape, "| test", Xte_p.shape, "| values between", Xtr_p.min(), "and", Xtr_p.max())

S1 -> kept 15 features: ['URLSimilarityIndex', 'LineOfCode', 'NoOfExternalRef', 'NoOfImage', 'NoOfSelfRef', 'NoOfJS', 'LargestLineLength', 'NoOfCSS', 'HasSocialNet', 'LetterRatioInURL', 'HasCopyrightInfo', 'HasDescription', 'NoOfOtherSpecialCharsInURL', 'IsHTTPS', 'SpacialCharRatioInURL']
   train (188296, 15) | test (47074, 15) | values between 0.0 and 1.0


S2 -> kept 30 features: ['LineOfCode', 'NoOfExternalRef', 'NoOfImage', 'NoOfSelfRef', 'NoOfJS', 'LargestLineLength', 'NoOfCSS', 'HasSocialNet', 'LetterRatioInURL', 'HasCopyrightInfo', 'HasDescription', 'NoOfOtherSpecialCharsInURL', 'IsHTTPS', 'SpacialCharRatioInURL', 'DomainTitleMatchScore', 'HasSubmitButton', 'URLTitleMatchScore', 'NoOfDegitsInURL', 'DegitRatioInURL', 'IsResponsive', 'NoOfEmptyRef', 'NoOfiFrame', 'URLLength', 'NoOfLettersInURL', 'HasHiddenFields', 'HasFavicon', 'HasTitle', 'DomainLength', 'Robots', 'Pay']
   train (188296, 30) | test (47074, 30) | values between 0.0 and 1.0


Now every feature is on a 0-1 scale and we have 15 (S1) or 30 (S2) features instead of 50 (S1) or 46 (S2).
The two prepared datasets are stored in prepared for the next task.